# Avaliação por pixel (estágio 11)

Executa a inferência dos pesos já treinados dos estágios 09 (U-Net) e 10 (SegFormer) sobre os patches de validação da divisão espacial da **fonte escolhida manualmente** (`ground_truth.chosen_source`, estágio 04). Para cada dobra e modelo, calcula IoU, F1, Precisão e Recall em nível de pixel, ignorando os pixels fora do AOI (rótulo `-1`), e persiste métricas agregadas, métricas por patch e predições binárias sob `MyDrive/tcc/artifacts/metrics/{modelo}/<fonte>/evaluation/`. O estágio não retreina: apenas avalia os artefatos existentes.


## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.


In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc-lab/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## Dependências pinadas

Garante as versões fixadas em `requirements-runtime.txt`, incluindo o `transformers` usado pelo SegFormer, com o mesmo conjunto nas duas plataformas. Quando uma biblioteca com extensão compilada já importada (numpy, scipy, torch, entre outras) diverge da versão pinada, o kernel é reiniciado automaticamente para evitar inconsistência de ABI; nesse caso, o notebook é reexecutado a partir da primeira célula. Também remove um `torchaudio` eventualmente incompatível (compilado para outra versão de CUDA), que o `transformers` importa indiretamente ao carregar modelos; o projeto não usa áudio.


In [ ]:
# Garante as versões pinadas, reinicia o kernel em divergência de ABI e remove
# torchaudio incompatível que impediria o import do transformers.
from src.setup import install_runtime_requirements, remove_incompatible_torchaudio

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
install_runtime_requirements(requirements)
remove_incompatible_torchaudio()


## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo modelos, métricas e figuras. A fonte avaliada é a mesma escolhida manualmente na fase de ground truth.


In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src.data.patch_generation import manifest_path
from src.data.source_decision import chosen_source
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
source_name = chosen_source()
print(f"Plataforma: {platform}")
print(f"Fonte de treinamento/avaliação (decisão manual): {source_name}")
print(f"Manifesto: {manifest_path(storage_paths, source_name)}")
print(f"Modelos: {storage_paths['models']}")
print(f"Métricas: {storage_paths['artifacts_metrics']}")
print(f"Figuras: {storage_paths['artifacts_figures']}")


## Reprodutibilidade e determinismo

Fixa as sementes de python/numpy/torch/cuda, habilita as flags determinísticas do PyTorch e registra as versões dos pacotes principais. A avaliação é somente inferência, sem retropropagação; ainda assim mantém `warn_only=True` para o SegFormer, coerente com o estágio 10, evitando interrupção por operações do modelo sem implementação determinística no CUDA. Sementes, cuDNN e a ordem dos lotes permanecem determinísticos.


In [ ]:
# Fixa sementes e flags determinísticas; warn_only cobre operações do SegFormer no CUDA.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config, warn_only=True)
log_environment()


## Dependência dos estágios 06, 07, 08, 09 e 10

Verifica que o manifesto de patches da fonte (estágio 06), a divisão espacial k-fold (estágio 07) e as estatísticas de normalização (estágio 08) estão disponíveis e vigentes. Confirma também que, para **cada modelo** e **cada dobra**, os pesos, as métricas e o histórico de treino dos estágios 09/10 existem, sem retreinar nada nesta etapa.


In [ ]:
# Verifica as dependências dos estágios anteriores e os artefatos de treino dos dois modelos.
from src.data.eda import normalization_stats_is_current, normalization_stats_path
from src.data.patch_generation import (
    load_manifest,
    manifest_has_folds,
    manifest_is_current,
    manifest_path,
)
from src.data.spatial_split import fold_count, split_is_current
from src.evaluation import MODEL_NAMES
from src.trainer import fold_training_done
from src.utils import check_dependencies

manifest_file = manifest_path(storage_paths, source_name)
check_dependencies(
    {
        "Manifesto (estágio 06)": manifest_file,
        "Estatísticas de normalização (estágio 08)": normalization_stats_path(
            storage_paths, source_name
        ),
    }
)
manifest = load_manifest(storage_paths, source_name)
if not manifest_has_folds(manifest):
    raise ValueError("Coluna fold ausente/incompleta; execute o estágio 07 antes.")
if not split_is_current(storage_paths, source_name):
    raise ValueError("Divisão espacial desatualizada; execute o estágio 07 antes.")
if not normalization_stats_is_current(storage_paths, source_name):
    raise ValueError("Estatísticas de normalização desatualizadas; execute o estágio 08 antes.")
if not manifest_is_current(storage_paths, source_name):
    raise ValueError("Manifesto desatualizado frente às entradas; execute o estágio 06 antes.")

for model_name in MODEL_NAMES:
    for fold in range(fold_count()):
        if not fold_training_done(storage_paths, model_name, source_name, fold):
            raise FileNotFoundError(
                f"Artefatos de treino ausentes para {model_name}, dobra {fold}. "
                "Execute o estágio 09 (U-Net) ou 10 (SegFormer) antes da avaliação."
            )
print(f"Patches: {len(manifest)} | Dobras: {fold_count()} | Modelos: {', '.join(MODEL_NAMES)}")


## Preparação local dos patches

Garante, no Kaggle, que todos os patches do manifesto da fonte tenham cópia no cache local antes da inferência, evitando requisições à Drive API durante o carregamento por lote; no-op no Colab/local.


In [ ]:
# Garante cópias locais dos patches (Kaggle); no-op no Colab/local.
from src.data.dataset import ensure_patches_local

local_files = ensure_patches_local(storage_paths, manifest)
print(f"Arquivos preparados localmente: {local_files}")


## Avaliação dos modelos por dobra

Percorre os dois modelos e todas as dobras da validação espacial. Para cada modelo/dobra, `src.evaluation.evaluate_fold` carrega os pesos treinados, executa a inferência sobre os patches de validação e persiste métricas agregadas (`fold_{i}.json`), métricas por patch (`patch_metrics_fold_{i}.parquet`) e predições binárias (`predictions_fold_{i}.npz`). Os artefatos são reutilizados apenas quando o fingerprint do manifesto, o fingerprint do split, o hash da configuração e o hash dos pesos coincidem; caso contrário, a dobra é reavaliada automaticamente. A célula exige GPU para viabilizar a inferência dos dois modelos em todas as dobras.


In [ ]:
# Avalia os dois modelos em todas as dobras e persiste os artefatos (reutiliza quando vigentes).
import torch

from src.evaluation import (
    MODEL_NAMES,
    evaluate_model,
    save_evaluation_figure,
    summarize_model,
)

# Exige GPU: a inferência de 512x512 por duas arquiteturas em todas as dobras é inviável em CPU.
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU indisponível: ative o acelerador de GPU (T4/P100) na plataforma antes de avaliar."
    )
device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print(f"Dispositivo: {device} | {torch.cuda.get_device_name(0)} | {props.total_memory / 1e9:.1f} GB")
results = {}
for model_name in MODEL_NAMES:
    print(f"\nAvaliando {model_name}...")
    evaluations = evaluate_model(storage_paths, model_name, device, source_name=source_name)
    summary = summarize_model(storage_paths, model_name, source_name, evaluations)
    figure = save_evaluation_figure(storage_paths, model_name, source_name, summary)
    results[model_name] = {"summary": summary, "figure": figure}


## Métricas por dobra e modelo

Exibe uma tabela com as métricas de pixel de cada dobra e de cada modelo, permitindo inspecionar a variabilidade entre as dobras espaciais antes da consolidação.


In [ ]:
# Monta a tabela de métricas por dobra e modelo.
import pandas as pd

rows = []
for model_name, payload in results.items():
    for item in payload["summary"]["per_fold"]:
        rows.append(
            {
                "modelo": model_name,
                "dobra": item["fold"],
                "patches": item["n_patches"],
                "iou": round(item["metrics"]["iou"], 4),
                "f1": round(item["metrics"]["f1"], 4),
                "precision": round(item["metrics"]["precision"], 4),
                "recall": round(item["metrics"]["recall"], 4),
            }
        )
pd.DataFrame(rows)


## Resumo consolidado e artefatos

Exibe o agregado *micro* (TP/FP/FN somados entre as dobras), a média e o desvio das métricas por dobra e os caminhos dos artefatos de cada modelo: resumo em `artifacts/metrics/{modelo}/<fonte>/evaluation/summary.json` e figura em `artifacts/figures/`.


In [ ]:
# Exibe o resumo consolidado e os caminhos dos artefatos de avaliação.
from src.evaluation import evaluation_summary_path
from src.utils import print_summary

summary = {}
for model_name, payload in results.items():
    model_summary = payload["summary"]
    aggregate = model_summary["aggregate"]["metrics"]
    fold_mean = model_summary["fold_mean"]
    fold_std = model_summary["fold_std"]
    summary[f"{model_name} — IoU agregado"] = f"{aggregate['iou']:.4f}"
    summary[f"{model_name} — F1 agregado"] = f"{aggregate['f1']:.4f}"
    summary[f"{model_name} — Precisão agregada"] = f"{aggregate['precision']:.4f}"
    summary[f"{model_name} — Recall agregado"] = f"{aggregate['recall']:.4f}"
    summary[f"{model_name} — IoU por dobra (média ± desvio)"] = (
        f"{fold_mean['iou']:.4f} ± {fold_std['iou']:.4f}"
    )
    summary[f"{model_name} — resumo"] = str(
        evaluation_summary_path(storage_paths, model_name, source_name)
    )
    summary[f"{model_name} — figura"] = str(payload["figure"])
print_summary(summary, "11")
